In [ ]:

import findspark
from pyspark.sql import SparkSession

findspark.init()
spark = SparkSession\
        .builder\
        .master("local[2]")\
        .appName("Spark MLlib")\
        .getOrCreate()
spark.sparkContext

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col
import matplotlib.pyplot as plt

In [ ]:
df = spark.read.csv("Otodom_Flat_Listings.csv", header=True, inferSchema=True)

In [ ]:
df_cleaned = df.dropna(subset=['Price', 'Surface', 'Number_of_Rooms', 'Voivodeship'])

df_cleaned = df_cleaned.withColumn("Surface", col("Surface").cast("float"))
df_cleaned = df_cleaned.withColumn("Number_of_Rooms", col("Number_of_Rooms").cast("int"))

df_cleaned = df_cleaned.fillna({'Surface': 0.0, 'Number_of_Rooms': 0})

indexer = StringIndexer(inputCol="Voivodeship", outputCol="Voivodeship_indexed", handleInvalid="skip")
encoder = OneHotEncoder(inputCol="Voivodeship_indexed", outputCol="Voivodeship_onehot")

assembler = VectorAssembler(inputCols=["Surface", "Number_of_Rooms", "Voivodeship_onehot"], outputCol="features", handleInvalid="skip")

lr = LinearRegression(featuresCol="features", labelCol="Price")

pipeline = Pipeline(stages=[indexer, encoder, assembler, lr])

train_data, test_data = df_cleaned.randomSplit([0.8, 0.2])

model = pipeline.fit(train_data)

predictions = model.transform(test_data)

predictions.select("features", "Price", "prediction").show()

evaluator_rmse = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="Price", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE: {rmse}")
print(f"R?: {r2}")

predicted_values = predictions.select("Price", "prediction").toPandas()

plt.figure(figsize=(10, 6))
plt.scatter(predicted_values['Price'], predicted_values['prediction'], color='blue', alpha=0.5)
plt.plot([min(predicted_values['Price']), max(predicted_values['Price'])],
         [min(predicted_values['Price']), max(predicted_values['Price'])],
         color='red', lw=2)
plt.xlabel('Rzeczywista cena')
plt.ylabel('Przewidywana cena')
plt.title('Por?wnanie rzeczywistej i przewidywanej ceny')
plt.show()